# Automatic fine-tuning

Run the automatic parameter grid outside the web interface. Pick a normal baseline model, train the grid, and persist the run summary to the registry database.

In [ ]:
import json
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "ai_module":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.registry import _connect, _ensure_schema
from backend.notebook_training import (
    automatic_model_specs,
    automatic_run_summaries,
    rank_models,
    run_automatic_fine_tuning,
)


In [ ]:
def normal_model_table():
    with _connect() as con:
        _ensure_schema(con)
        rows = con.execute(
            """
            SELECT model_id, algorithm, feature_set, uncertainty_variant, balancing_method, metrics
            FROM _model_results
            WHERE model_id NOT LIKE '%__tuned_%'
            ORDER BY model_id
            """
        ).fetchall()
    records = []
    for row in rows:
        metrics = json.loads(row["metrics"])
        records.append({
            "model_id": row["model_id"],
            "algorithm": row["algorithm"],
            "feature_set": row["feature_set"],
            "uncertainty_variant": row["uncertainty_variant"],
            "balancing_method": row["balancing_method"],
            "auc": metrics.get("auc"),
            "f1": metrics.get("f1"),
            "recall": metrics.get("recall"),
        })
    return pd.DataFrame(records)

models = normal_model_table()
models


## Select the baseline

Set `BASE_MODEL_ID` to a normal model id from the table above. If left blank, the first available normal model is used.

In [ ]:
BASE_MODEL_ID = ""
USE_GPU = True
PERSIST_RUN = True

if not BASE_MODEL_ID:
    if models.empty:
        raise ValueError("No normal models are registered. Run train_data_balanced_models.ipynb or train_models.ipynb first.")
    BASE_MODEL_ID = str(models.iloc[0]["model_id"])

base_model, specs = automatic_model_specs(BASE_MODEL_ID)
print(f"Selected baseline: {BASE_MODEL_ID}")
print(f"Automatic combinations: {len(specs)}")


In [ ]:
job = run_automatic_fine_tuning(
    BASE_MODEL_ID,
    persist=PERSIST_RUN,
    use_gpu=USE_GPU,
)
job["status"], job["message"], job["result"]["trained"], job["result"]["reused"]


In [ ]:
ranked = rank_models(job["result"]["models"])
pd.DataFrame([
    {
        "rank": index,
        "model_id": model["modelId"],
        "threshold": model["classificationThreshold"],
        "auc": model["metrics"].get("auc"),
        "f1": model["metrics"].get("f1"),
        "recall": model["metrics"].get("recall"),
        **model.get("hyperparameters", {}),
    }
    for index, model in enumerate(ranked, start=1)
]).head(20)


In [ ]:
pd.DataFrame(automatic_run_summaries()).head(10)
